In [4]:
import pandas as pd

# Load the IMDB dataset, using the 'python' engine for better handling of parsing errors
df = pd.read_csv('/content/IMDB Dataset.csv', engine='python')

# Display the first 5 rows
print("First 5 rows of the DataFrame:")
print(df.head())

# Print the shape of the DataFrame
print("\nShape of the DataFrame:")
print(df.shape)

# Display concise summary of the DataFrame
print("\nConcise summary of the DataFrame:")
df.info()

First 5 rows of the DataFrame:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Shape of the DataFrame:
(50000, 2)

Concise summary of the DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [7]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# 1. Download necessary NLTK data
print("Downloading NLTK data (stopwords, punkt, wordnet, punkt_tab)...")

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    # Added specific download for 'punkt_tab' as suggested by the error message
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

print("NLTK data download complete.")

# Initialize stemmer and lemmatizer globally
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# 2. Define preprocess_text function
def preprocess_text(text):
    # a. Convert to lowercase
    text = text.lower()
    # b. Remove HTML tags (e.g., <br />)
    text = re.sub(r'<.*?>', '', text)
    # c. Remove punctuation and special characters (keep alphanumeric and spaces)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # d. Tokenize the cleaned text
    tokens = word_tokenize(text)
    # e. Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # f. Join tokens back into a string (for cleaned_review column)
    return ' '.join(tokens)

# 3. Apply the preprocess_text function
print("\nApplying text preprocessing to 'review' column...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Text preprocessing complete.")

# 4. Define tokenize_text function
def tokenize_text(text):
    return word_tokenize(text)

# 5. Apply the tokenize_text function
print("\nTokenizing cleaned reviews...")
df['tokenized_review'] = df['cleaned_review'].apply(tokenize_text)
print("Tokenization complete.")

# 6. Instantiate stemmer and lemmatizer (already done globally)

# 7. Define apply_stemming and apply_lemmatization functions
def apply_stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

# 8. Apply the apply_stemming function
print("\nApplying stemming...")
df['stemmed_review'] = df['tokenized_review'].apply(apply_stemming)
print("Stemming complete.")

# 9. Apply the apply_lemmatization function
print("\nApplying lemmatization...")
df['lemmatized_review'] = df['tokenized_review'].apply(apply_lemmatization)
print("Lemmatization complete.")

# 10. Select a sample original review and compare
sample_index = 0
original_review = df['review'].iloc[sample_index]
cleaned_tokenized_review = df['tokenized_review'].iloc[sample_index]
stemmed_review = df['stemmed_review'].iloc[sample_index]
lemmatized_review = df['lemmatized_review'].iloc[sample_index]

print(f"\n--- Sample Text Preprocessing Comparison (Index: {sample_index}) ---")
print(f"Original Review:\n{original_review}")
print(f"\nCleaned & Tokenized Review:\n{cleaned_tokenized_review}")
print(f"\nStemmed Review:\n{stemmed_review}")
print(f"\nLemmatized Review:\n{lemmatized_review}")

print("\n--- Analysis of Stemming vs. Lemmatization ---")
print("Stemming typically reduces words to their root form by chopping off suffixes, often resulting in words that are not actual dictionary words (e.g., 'univers' from 'university', 'mani' from 'many'). It's a faster, rule-based approach.")
print("Lemmatization, on the other hand, reduces words to their base or dictionary form (lemma), ensuring that the resulting word is a valid word (e.g., 'university' from 'universities', 'man' from 'men'). It's typically slower as it involves a dictionary lookup and morphological analysis.")
print("In the sample above, we can observe how words might be truncated by stemming (e.g., 'reviewers' -> 'review') while lemmatization attempts to return a valid dictionary word. For example, 'mentioned' might become 'mention' in lemmatization, but 'mention' with stemming. The exact output depends on the specific words and the NLTK algorithms.")


NLTK data download complete.

Applying text preprocessing to 'review' column...
Text preprocessing complete.

Tokenizing cleaned reviews...
Tokenization complete.

Applying stemming...
Stemming complete.

Applying lemmatization...
Lemmatization complete.

--- Sample Text Preprocessing Comparison (Index: 0) ---
Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cel

In [8]:
from nltk.util import ngrams
from collections import Counter

# 1. Import the ngrams function from nltk.util and collections.Counter (done above)

# 2. Define a function generate_ngrams(tokens, n)
def generate_ngrams(tokens, n):
    return list(ngrams(tokens, n))

# 3. Apply this function to the df['tokenized_review'] column
print("\nGenerating unigrams...")
df['unigrams'] = df['tokenized_review'].apply(lambda x: generate_ngrams(x, 1))
print("Generating bigrams...")
df['bigrams'] = df['tokenized_review'].apply(lambda x: generate_ngrams(x, 2))
print("Generating trigrams...")
df['trigrams'] = df['tokenized_review'].apply(lambda x: generate_ngrams(x, 3))
print("N-gram generation complete.")

# 4. Select a sample review and print its original cleaned_review string, and its generated unigrams, bigrams, and trigrams.
sample_index = 0 # Using the same sample_index as previous step
sample_cleaned_review = df['cleaned_review'].iloc[sample_index]
sample_unigrams = df['unigrams'].iloc[sample_index]
sample_bigrams = df['bigrams'].iloc[sample_index]
sample_trigrams = df['trigrams'].iloc[sample_index]

print(f"\n--- Sample N-gram Representation (Index: {sample_index}) ---")
print(f"Cleaned Review (string):\n{sample_cleaned_review}")
print(f"\nUnigrams (n=1):\n{sample_unigrams[:10]}...") # Displaying first 10 for brevity
print(f"\nBigrams (n=2):\n{sample_bigrams[:10]}...") # Displaying first 10 for brevity
print(f"\nTrigrams (n=3):\n{sample_trigrams[:10]}...") # Displaying first 10 for brevity

print("\n--- Analysis of N-gram Representation ---")
print("Unigrams represent individual words, providing a basic bag-of-words view of the text. They capture the presence of words but lose sequential information. For example, 'one' and 'reviewers' are individual tokens.")
print("Bigrams represent pairs of adjacent words. They capture short-range word dependencies and local context. For instance, ('one', 'reviewers') or ('reviewers', 'mentioned') give more meaning than individual words.")
print("Trigrams represent sequences of three adjacent words. They capture even more localized context and word order than bigrams, helping to differentiate phrases with similar words but different meanings (e.g., ('reviewers', 'mentioned', 'watching')). Higher-order N-grams tend to be sparser but offer richer contextual information about the text.")


Generating unigrams...
Generating bigrams...
Generating trigrams...
N-gram generation complete.

--- Sample N-gram Representation (Index: 0) ---
Cleaned Review (string):
one reviewers mentioned watching 1 oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use wordit called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy high agenda em city home manyaryans muslims gangstas latinos christians italians irish moreso scuffles death stares dodgy dealings shady agreements never far awayi would say main appeal show due fact goes shows wouldnt dare forget pretty pictures painted mainstream audiences forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched 

In [9]:
import collections
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

# 5. Build a vocabulary based on these N-grams
print("\nBuilding N-gram vocabularies...")

# Flatten all unigrams, bigrams, and trigrams
all_unigrams = [item for sublist in df['unigrams'] for item in sublist]
all_bigrams = [item for sublist in df['bigrams'] for item in sublist]
all_trigrams = [item for sublist in df['trigrams'] for item in sublist]

# Create sets of unique items for each type to get the vocabulary
unigram_vocabulary = set(all_unigrams)
bigram_vocabulary = set(all_bigrams)
trigram_vocabulary = set(all_trigrams)

# Print the size of each unique N-gram vocabulary
print(f"Size of Unigram Vocabulary: {len(unigram_vocabulary)}")
print(f"Size of Bigram Vocabulary: {len(bigram_vocabulary)}")
print(f"Size of Trigram Vocabulary: {len(trigram_vocabulary)}")
print("N-gram vocabulary building complete.")

# 6. Calculate simple N-gram probabilities for a small text sample
print("\nCalculating N-gram probabilities for a sample text...")

sample_text_for_prob = "This movie was absolutely amazing and I loved every minute of it. It had a great plot and fantastic acting."

# Tokenize the small text (using the same preprocessing logic for consistency)
def preprocess_single_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

sample_tokens = preprocess_single_text(sample_text_for_prob)

print(f"Sample Text: {sample_text_for_prob}")
print(f"Tokenized Sample: {sample_tokens}")

# Generate N-grams for the sample text
sample_unigrams = list(ngrams(sample_tokens, 1))
sample_bigrams = list(ngrams(sample_tokens, 2))
sample_trigrams = list(ngrams(sample_tokens, 3))

# Calculate frequencies using collections.Counter
unigram_counts = collections.Counter(sample_unigrams)
bigram_counts = collections.Counter(sample_bigrams)
trigram_counts = collections.Counter(sample_trigrams)

# Display frequencies and probabilities
print("\n--- Sample N-gram Frequencies and Probabilities ---")

print("\nUnigrams:")
total_unigrams = len(sample_unigrams)
for gram, count in unigram_counts.most_common():
    print(f"  {gram}: Count={count}, Probability={count/total_unigrams:.4f}")

print("\nBigrams:")
total_bigrams = len(sample_bigrams)
if total_bigrams > 0:
    for gram, count in bigram_counts.most_common():
        print(f"  {gram}: Count={count}, Probability={count/total_bigrams:.4f}")
else:
    print("  No bigrams generated.")

print("\nTrigrams:")
total_trigrams = len(sample_trigrams)
if total_trigrams > 0:
    for gram, count in trigram_counts.most_common():
        print(f"  {gram}: Count={count}, Probability={count/total_trigrams:.4f}")
else:
    print("  No trigrams generated.")

print("\nN-gram probability calculation complete.")


Building N-gram vocabularies...
Size of Unigram Vocabulary: 221459
Size of Bigram Vocabulary: 3319005
Size of Trigram Vocabulary: 5487156
N-gram vocabulary building complete.

Calculating N-gram probabilities for a sample text...
Sample Text: This movie was absolutely amazing and I loved every minute of it. It had a great plot and fantastic acting.
Tokenized Sample: ['movie', 'absolutely', 'amazing', 'loved', 'every', 'minute', 'great', 'plot', 'fantastic', 'acting']

--- Sample N-gram Frequencies and Probabilities ---

Unigrams:
  ('movie',): Count=1, Probability=0.1000
  ('absolutely',): Count=1, Probability=0.1000
  ('amazing',): Count=1, Probability=0.1000
  ('loved',): Count=1, Probability=0.1000
  ('every',): Count=1, Probability=0.1000
  ('minute',): Count=1, Probability=0.1000
  ('great',): Count=1, Probability=0.1000
  ('plot',): Count=1, Probability=0.1000
  ('fantastic',): Count=1, Probability=0.1000
  ('acting',): Count=1, Probability=0.1000

Bigrams:
  ('movie', 'absolute

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Instantiate CountVectorizer
# max_features can be set to limit the vocabulary size, or left as None to use all unique words.
# Here, we'll use all unique words to see the full vocabulary size initially.
count_vectorizer = CountVectorizer()

# 2. Fit the vectorizer to the 'cleaned_review' column and transform it into a sparse matrix
print("Applying CountVectorizer (Bag of Words)...")
X_bow = count_vectorizer.fit_transform(df['cleaned_review'])
print("CountVectorizer applied.")

# 3. Print the shape of the X_bow matrix to report the number of documents and the vocabulary size
print(f"\nShape of X_bow matrix: {X_bow.shape}")
print(f"Number of documents: {X_bow.shape[0]}")
print(f"Vocabulary size (number of unique words): {X_bow.shape[1]}")

# 4. To show a sample vector output, convert a small portion of the X_bow matrix
# (e.g., the vector for the first review) to a dense array and print it.
sample_review_index = 0
sample_bow_vector = X_bow[sample_review_index].toarray()
print(f"\nSample BoW vector for review at index {sample_review_index} (first 20 elements):\n{sample_bow_vector[0][:20]}")

# Optionally, display some of the feature names (words in vocabulary)
feature_names = count_vectorizer.get_feature_names_out()
print(f"\nSample features (words in vocabulary): {feature_names[500:510]}")

Applying CountVectorizer (Bag of Words)...
CountVectorizer applied.

Shape of X_bow matrix: (50000, 221431)
Number of documents: 50000
Vocabulary size (number of unique words): 221431

Sample BoW vector for review at index 0 (first 20 elements):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Sample features (words in vocabulary): ['121566' '1216' '121699' '121701' '121am' '122' '12206' '12211'
 '12242009' '122452']


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. Define features (X) and target variable (y)
X = X_bow  # X_bow was created in the previous step using CountVectorizer
y = df['sentiment']

# 2. Convert 'sentiment' column to numerical labels
# 'positive' -> 1, 'negative' -> 0
y_encoded = y.map({'positive': 1, 'negative': 0})

print(f"Original sentiment labels (first 5): {y.head().tolist()}")
print(f"Encoded sentiment labels (first 5): {y_encoded.head().tolist()}")

# 3. Split the data into training and testing sets
# test_size=0.2 means 20% of data will be used for testing, 80% for training
# random_state ensures reproducibility of the split
print("\nSplitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# 4. Instantiate a Multinomial Naive Bayes classifier
print("\nInstantiating Multinomial Naive Bayes classifier...")
naive_bayes_classifier = MultinomialNB()

# 5. Train the Multinomial Naive Bayes classifier
print("Training the classifier...")
naive_bayes_classifier.fit(X_train, y_train)
print("Classifier training complete.")

# 6. Predict sentiment on the test data
print("\nPredicting sentiment on the test data...")
y_pred = naive_bayes_classifier.predict(X_test)
print("Prediction complete.")

# Display the first few predictions
print(f"\nFirst 10 predicted labels: {y_pred[:10].tolist()}")
print(f"First 10 actual labels (test set): {y_test[:10].tolist()}")

Original sentiment labels (first 5): ['positive', 'positive', 'positive', 'negative', 'positive']
Encoded sentiment labels (first 5): [1, 1, 1, 0, 1]

Splitting data into training and testing sets...
Shape of X_train: (40000, 221431)
Shape of X_test: (10000, 221431)
Shape of y_train: (40000,)
Shape of y_test: (10000,)

Instantiating Multinomial Naive Bayes classifier...
Training the classifier...
Classifier training complete.

Predicting sentiment on the test data...
Prediction complete.

First 10 predicted labels: [1, 1, 0, 1, 0, 1, 1, 0, 0, 0]
First 10 actual labels (test set): [1, 1, 0, 1, 0, 1, 1, 1, 0, 0]


In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
import numpy as np

# 1. Evaluate the currently trained Naive Bayes model (unigram BoW)
print("\n--- Evaluation of Unigram BoW Naive Bayes Model ---")

# a. Calculate and print the accuracy score
accuracy_uni = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy_uni:.4f}")

# b. Calculate and print the precision score
precision_uni = precision_score(y_test, y_pred)
print(f"Precision Score: {precision_uni:.4f}")

# c. Calculate and print the recall score
recall_uni = recall_score(y_test, y_pred)
print(f"Recall Score: {recall_uni:.4f}")

# d. Calculate and print the F1-score
f1_uni = f1_score(y_test, y_pred)
print(f"F1-Score: {f1_uni:.4f}")

# e. Generate and print the confusion matrix
conf_matrix_uni = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix_uni)

# f. Select 5-10 random indices from X_test and print original review, actual, predicted sentiments
print("\n--- Sample Predictions for Unigram BoW Model ---")
sample_indices = np.random.choice(y_test.index, 10, replace=False)

for i, idx in enumerate(sample_indices):
    original_review_text = df['review'].loc[idx]
    actual_sentiment = y_test.loc[idx]
    predicted_sentiment = y_pred[y_test.index.get_loc(idx)] # Get prediction for this index from y_pred

    # Convert numerical labels back to 'positive'/'negative' for better readability
    actual_sentiment_str = 'positive' if actual_sentiment == 1 else 'negative'
    predicted_sentiment_str = 'positive' if predicted_sentiment == 1 else 'negative'

    prediction_status = "Correct" if actual_sentiment == predicted_sentiment else "Incorrect"

    print(f"\nSample {i+1} (Index: {idx}):")
    print(f"  Original Review: {original_review_text[:100]}...") # Truncate for brevity
    print(f"  Actual Sentiment: {actual_sentiment_str}")
    print(f"  Predicted Sentiment: {predicted_sentiment_str} ({prediction_status})")

# 2. Prepare data for a new model using unigrams and bigrams
print("\n--- Preparing Data for Unigram + Bigram Model ---")

# a. Instantiate a new CountVectorizer with ngram_range=(1, 2)
count_vectorizer_bigram = CountVectorizer(ngram_range=(1, 2))

# b. Fit this new vectorizer to df['cleaned_review'] and transform it
X_bow_bigram = count_vectorizer_bigram.fit_transform(df['cleaned_review'])

print(f"Shape of X_bow_bigram matrix: {X_bow_bigram.shape}")
print(f"Vocabulary size (unigrams+bigrams): {X_bow_bigram.shape[1]}")

# c. Split X_bow_bigram and y_encoded into new training and testing sets
X_train_bigram, X_test_bigram, y_train_bigram, y_test_bigram = train_test_split(
    X_bow_bigram, y_encoded, test_size=0.2, random_state=42)

print(f"Shape of X_train_bigram: {X_train_bigram.shape}")
print(f"Shape of X_test_bigram: {X_test_bigram.shape}")

# 3. Train and evaluate a new Multinomial Naive Bayes model with unigrams and bigrams
print("\n--- Training and Evaluating Unigram + Bigram Naive Bayes Model ---")

# a. Instantiate a new MultinomialNB classifier
naive_bayes_classifier_bigram = MultinomialNB()

# b. Train naive_bayes_classifier_bigram
naive_bayes_classifier_bigram.fit(X_train_bigram, y_train_bigram)
print("Unigram + Bigram Classifier training complete.")

# c. Predict sentiment on X_test_bigram
y_pred_bigram = naive_bayes_classifier_bigram.predict(X_test_bigram)
print("Prediction with Unigram + Bigram model complete.")

# d. Calculate and print the accuracy, precision, recall, and F1-score for this model
accuracy_uni_bi = accuracy_score(y_test_bigram, y_pred_bigram)
precision_uni_bi = precision_score(y_test_bigram, y_pred_bigram)
recall_uni_bi = recall_score(y_test_bigram, y_pred_bigram)
f1_uni_bi = f1_score(y_test_bigram, y_pred_bigram)

print(f"\nAccuracy Score (Unigram + Bigram): {accuracy_uni_bi:.4f}")
print(f"Precision Score (Unigram + Bigram): {precision_uni_bi:.4f}")
print(f"Recall Score (Unigram + Bigram): {recall_uni_bi:.4f}")
print(f"F1-Score (Unigram + Bigram): {f1_uni_bi:.4f}")

# Store metrics for comparison in the next markdown cell
uni_metrics = {'Accuracy': accuracy_uni, 'Precision': precision_uni, 'Recall': recall_uni, 'F1-Score': f1_uni}
uni_bi_metrics = {'Accuracy': accuracy_uni_bi, 'Precision': precision_uni_bi, 'Recall': recall_uni_bi, 'F1-Score': f1_uni_bi}



--- Evaluation of Unigram BoW Naive Bayes Model ---
Accuracy Score: 0.8612
Precision Score: 0.8749
Recall Score: 0.8454
F1-Score: 0.8599
Confusion Matrix:
 [[4352  609]
 [ 779 4260]]

--- Sample Predictions for Unigram BoW Model ---

Sample 1 (Index: 40908):
  Original Review: The only reason to see this film is Sung Hi Lee, the stunning model/actress from Korea who plays "Mu...
  Actual Sentiment: negative
  Predicted Sentiment: negative (Correct)

Sample 2 (Index: 7678):
  Original Review: A powerful movie that has recovered much of its meaning in this second half of 2007 after the new de...
  Actual Sentiment: positive
  Predicted Sentiment: positive (Correct)

Sample 3 (Index: 7):
  Original Review: This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 ...
  Actual Sentiment: negative
  Predicted Sentiment: negative (Correct)

Sample 4 (Index: 772):
  Original Review: This movie is an amazing comedy.. the script is too funny.. if u wat

### Comparison of Model Performance (Unigram vs. Unigram + Bigram)

| Metric    | Unigram BoW Model | Unigram + Bigram BoW Model |
|-----------|-------------------|----------------------------|
| Accuracy  | 0.8612            | 0.8874                     |
| Precision | 0.8749            | 0.9012                     |
| Recall    | 0.8454            | 0.8722                     |
| F1-Score  | 0.8599            | 0.8864                     |

**Discussion:**

From the comparison, it's evident that the Unigram + Bigram BoW Model significantly outperforms the Unigram BoW Model across all evaluated metrics: Accuracy, Precision, Recall, and F1-Score.

**Observed Differences and Hypotheses:**

1.  **Improved Performance:** The addition of bigrams (pairs of consecutive words) to the feature set has led to a noticeable improvement in the model's ability to classify sentiment. This is a common observation in NLP tasks.

2.  **Increased Context:** Unigrams (single words) treat each word independently, ignoring word order and local context. For example,

### Comparison of Model Performance (Unigram vs. Unigram + Bigram)

| Metric    | Unigram BoW Model | Unigram + Bigram BoW Model |
|-----------|-------------------|----------------------------|
| Accuracy  | 0.8612            | 0.8874                     |
| Precision | 0.8749            | 0.9012                     |
| Recall    | 0.8454            | 0.8722                     |
| F1-Score  | 0.8599            | 0.8864                     |

**Discussion:**

From the comparison, it's evident that the **Unigram + Bigram BoW Model significantly outperforms the Unigram BoW Model** across all evaluated metrics: Accuracy, Precision, Recall, and F1-Score.

**Observed Differences and Hypotheses:**

1.  **Improved Performance:** The addition of bigrams (pairs of consecutive words) to the feature set has led to a noticeable improvement in the model's ability to classify sentiment. This is a common observation in NLP tasks.

2.  **Increased Context:** Unigrams (single words) treat each word independently, ignoring word order and local context. For example, "not good" conveys a different sentiment than "good", but a unigram model might only count "not" and "good" separately. Bigrams, like ("not", "good"), capture this local context, allowing the model to differentiate more nuanced sentiments. This richer contextual information helps the model make more accurate classifications.

3.  **Increased Dimensionality and Sparsity:** While bigrams provide more context, they also dramatically increase the vocabulary size (from 221,431 for unigrams to 3,530,972 for unigrams+bigrams). This leads to an even higher-dimensional and sparser feature space. Despite this increase in complexity and sparsity, the Multinomial Naive Bayes model was able to leverage the added contextual information effectively, indicating that the benefits of richer features outweighed the potential drawbacks of increased sparsity for this dataset and model.

### Q&A

1.  **How did stemming/lemmatization affect the model?**
    Stemming and lemmatization primarily aim to reduce vocabulary size by normalizing different forms of a word. While not directly used in the final BoW vectorization, if applied, lemmatization (which reduces words to their dictionary form) would better preserve semantic meaning than stemming (which often produces non-dictionary root forms). For Naive Bayes, the main benefit is vocabulary reduction, which can lead to a less sparse feature matrix and improve statistical reliability by aggregating counts for conceptually similar words.

2.  **Did N-grams improve model performance?**
    Yes, the inclusion of bigrams alongside unigrams significantly improved the model's performance across all evaluation metrics. The Unigram + Bigram BoW Model achieved an Accuracy of 0.8874 and an F1-Score of 0.8864, compared to the Unigram BoW Model's Accuracy of 0.8612 and F1-Score of 0.8599. This improvement is attributed to bigrams providing crucial local context, helping the model capture more nuanced sentiment (e.g., differentiating "good" from "not good").

3.  **How did vectorization choices (Bag of Words using CountVectorizer) influence the results?**
    CountVectorizer transformed the text into a high-dimensional (unigram vocabulary of 221,431; unigram+bigram vocabulary of 3,530,972) and highly sparse numerical representation based on word frequencies. This approach influenced results positively as Multinomial Naive Bayes models perform well with frequency-based features and are robust to high dimensionality and sparsity. However, BoW's fundamental limitation is its disregard for word order and deeper semantic meaning, although N-grams partially mitigate the word order issue.

4.  **Why is Naive Bayes generally effective for text classification tasks?**
    Naive Bayes classifiers are effective for text classification due to their probabilistic nature, computational efficiency, and ability to handle high-dimensional, sparse data typical of text. Despite its "naive" assumption of conditional independence between words, it performs well by leveraging word frequencies. It's simple to implement, fast to train, relatively robust to irrelevant features, and can work with moderate amounts of training data, making it a strong baseline for many NLP tasks.


If more time were available, several avenues could be explored to potentially improve the sentiment classification model and deepen the analysis:

Explore TF-IDF Vectorization: Instead of raw word counts from CountVectorizer, apply TfidfVectorizer. TF-IDF (Term Frequency-Inverse Document Frequency) weights words based on their importance to a document in a collection or corpus, penalizing common words that appear across many documents (like 'movie') and boosting rarer, more discriminative words. This could improve performance by giving more weight to semantically significant terms.

Hyperparameter Tuning: Systematically tune the hyperparameters of the MultinomialNB classifier and CountVectorizer (e.g., alpha for Naive Bayes, min_df, max_df, max_features for vectorizers). Techniques like GridSearchCV or RandomizedSearchCV could be used to find the optimal combination of parameters.

Advanced Preprocessing:

Part-of-Speech (POS) tagging for Lemmatization: Currently, NLTK's WordNetLemmatizer lemmatizes without POS tags, assuming nouns by default. Providing the correct POS tag (e.g., verb, adjective) can lead to more accurate lemmatization (e.g., 'better' -> 'good' with adjective tag, 'meeting' -> 'meet' with verb tag). This could refine the vocabulary.
Handling Negation: BoW and N-grams still struggle with negation (e.g., 'not good' vs. 'good'). Custom preprocessing to identify and handle negation (e.g., appending '_NEG' to words following a negation) could significantly improve sentiment analysis.
Word Embeddings (e.g., Word2Vec, GloVe, FastText): These techniques represent words as dense vectors in a continuous vector space, capturing semantic relationships between words. Words with similar meanings are located closer in the vector space. This moves beyond the BoW limitation of treating words as independent entities and can capture nuanced meaning.

Deep Learning Models: For more sophisticated text representation and classification:

Recurrent Neural Networks (RNNs) / LSTMs / GRUs: These models can process sequences of words, explicitly capturing word order and long-range dependencies, which is critical for understanding complex sentences.
Convolutional Neural Networks (CNNs): While often used for image data, CNNs can also be effective for text classification by identifying local patterns (like N-grams) in word embeddings.
Transformer Models (e.g., BERT, GPT): State-of-the-art models that leverage self-attention mechanisms to capture highly complex contextual relationships between words, often leading to significant performance gains on various NLP tasks, including sentiment analysis.
Ensemble Methods: Combine predictions from multiple models (e.g., Naive Bayes, Logistic Regression, SVM) or different vectorization schemes to potentially achieve higher accuracy and robustness.